# Alerta Chuva SP · Visão Computacional — **versão 2**
**GS 2026.2 · CitizenTech · Applied Computer Vision**

A versão 1 acertava bem a **rua alagada** (mAP50 89%), mas era fraca em **veículo ilhado** e **pessoa na água**: havia poucos exemplos no treino e só 3 pessoas e 5 veículos no teste. Nesta versão:

1. **Mais dados das classes fracas:** entram mais 4 datasets do Roboflow com carros e pessoas em enchentes.
2. **Pseudo-rotulagem da água:** esses datasets não desenham caixa na água. O modelo v1 rotula a água neles automaticamente, só com confiança ≥ 0,50 (aprendizado semi-supervisionado).
3. **Novo split por foto original, estratificado:** as cópias aumentadas da mesma foto nunca ficam em splits diferentes, e o teste recebe pessoas e veículos suficientes para a métrica valer.
4. **Comparação justa:** v1 e v2 são avaliados no **mesmo** teste novo.

Como rodar: GPU T4, **Secrets** com `ROBOFLOW_API_KEY` e depois `Executar tudo`. Leva cerca de 1h30 a 2h; **deixe a aba aberta**.

## 1. Ambiente

In [ ]:
!nvidia-smi -L
%pip install -q ultralytics roboflow onnx onnxslim
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| GPU:", torch.cuda.is_available())

## 2. Código do projeto (com o modelo v1 em `modelos/best.pt`)

In [ ]:
REPO_URL = "https://github.com/zBaldd/GS-VISAO-COMPUTACIONAL.git"
import os, shutil
if not os.path.exists("/content/repo"):
    !git clone -q $REPO_URL /content/repo
%cd /content/repo
!git pull -q
os.makedirs("modelos", exist_ok=True)
shutil.copy2("modelos/best.pt", "modelos/v1_best.pt")   # guarda o v1 para a comparação
!ls modelos

### 2.1 Salvar o treino no Google Drive (recomendado)
Se o Colab desconectar no meio do treino, os pesos ficam salvos no Drive e dá para continuar. Autorize o acesso quando ele pedir.

In [ ]:
SALVAR_NO_DRIVE = True
RUNS = "/content/runs"
if SALVAR_NO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNS = "/content/drive/MyDrive/gs_visao_runs"
os.makedirs(RUNS, exist_ok=True)
print("Treinos em:", RUNS)

## 3. Dados: 7 datasets do Roboflow Universe (CC BY 4.0)

| Dataset | Por que entra | Opções |
|---|---|---|
| flooding-object-detection-opiei v3 | água, carro e pessoa (já usado na v1) | — |
| flood-detection-aicourse v4 | rua alagada e carro (v1) | — |
| flood-nfc4z v1 | água (v1) | — |
| flooded-objects v15 | **flooded car, human in flood** | não rotula água → pseudo-rótulo |
| flood-cka5z v1 (8,9 mil fotos) | **car e person em ruas alagadas** | "car"→veículo ilhado, "person"→pessoa na água; até 1.500 fotos para não dominar o resto; pseudo-rótulo |
| flood-object-detection-2olqp v1 | **flooded_car, person, person_in_help** | "person"→pessoa na água; pseudo-rótulo |
| flood-person-detection-uu8x0 v1 | **person_in_water** | pessoas em barco ou fora da água são descartadas; pseudo-rótulo |

In [ ]:
from google.colab import userdata
from roboflow import Roboflow
import json

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))

DATASETS = [
    # workspace, projeto, versão, opções (mapa próprio, limite de fotos, sem rótulo de água)
    ("-alpdg", "flooding-object-detection-opiei", 3, {}),
    ("ai-course-4kqzb", "flood-detection-aicourse", 4, {}),
    ("ambitious-jda7x", "flood-nfc4z", 1, {}),
    ("sahar-turki-dhamad", "flooded-objects", 15, {"sem_agua": True}),
    ("deep-learning-dohjx", "flood-cka5z", 1, {"mapa": {"car": "veiculo_ilhado", "person": "pessoa_na_agua", "house": None},
                                               "max_imagens": 1500, "sem_agua": True}),
    ("columbia-university-d9ggz", "flood-object-detection-2olqp", 1, {"mapa": {"person": "pessoa_na_agua"}, "sem_agua": True}),
    ("intern-ijiuy", "flood-person-detection-uu8x0", 1, {"sem_agua": True}),
]

config = []
for ws, proj, ver, opc in DATASETS:
    destino = f"/content/datasets/{proj}"
    try:
        if not os.path.exists(f"{destino}/data.yaml"):
            rf.workspace(ws).project(proj).version(ver).download("yolov8", location=destino)
        config.append({"pasta": destino, "nome": proj, **opc})
        print("OK     ", proj)
    except Exception as e:
        print("FALHOU ", proj, "->", e)

json.dump(config, open("/content/datasets.json", "w"), indent=2)

os.makedirs("docs", exist_ok=True)
linhas = ["# Datasets utilizados (Roboflow Universe, CC BY 4.0)", ""]
for ws, proj, ver, _ in DATASETS:
    linhas.append(f"- {proj} v{ver}, workspace `{ws}`: https://universe.roboflow.com/{ws}/{proj}")
open("docs/creditos_datasets.md", "w").write(chr(10).join(linhas) + chr(10))
print(len(config), "de", len(DATASETS), "datasets prontos")

## 4. Dataset unificado com novo split (75% / 12,5% / 12,5%)

In [ ]:
!python src/preparar_dataset.py --config /content/datasets.json --saida /content/dados/alagamento --resplit 0.75,0.125,0.125 --docs docs --sufixo _v2

## 5. Pseudo-rotulagem da água com o modelo v1
Só nas imagens dos datasets marcados com `sem_agua`, só caixas com confiança ≥ 0,50, e sem mexer nos rótulos humanos de carro e pessoa.

In [ ]:
!python src/pseudo_rotular.py --dados /content/dados/alagamento --modelo modelos/v1_best.pt --conf 0.5 --docs docs/resultados_v2

In [ ]:
# Conferência visual: amostras pseudo-rotuladas (azul = água do modelo v1; laranja/vermelho = rótulo humano)
import random, cv2, matplotlib.pyplot as plt
from pathlib import Path
CL = ["area_alagada", "veiculo_ilhado", "pessoa_na_agua"]; COR = [(30,140,230),(255,165,0),(220,40,40)]
raiz = Path("/content/dados/alagamento")
lista = [l for l in (raiz / "pseudo_rotuladas.txt").read_text().splitlines() if l]
random.seed(3); amostra = random.sample(lista, min(8, len(lista)))
fig, ax = plt.subplots(2, 4, figsize=(18, 8))
for a, rel in zip(ax.flat, amostra):
    p = raiz / rel; im = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB); h, w = im.shape[:2]
    for ln in (p.parent.parent / "labels" / (p.stem + ".txt")).read_text().splitlines():
        c, cx, cy, bw, bh = ln.split(); c = int(c); cx, cy, bw, bh = map(float, (cx, cy, bw, bh))
        cv2.rectangle(im, (int((cx-bw/2)*w), int((cy-bh/2)*h)), (int((cx+bw/2)*w), int((cy+bh/2)*h)), COR[c], max(2, w//200))
    a.imshow(im); a.axis("off")
plt.tight_layout(); os.makedirs("docs/img", exist_ok=True); plt.savefig("docs/img/pseudo_rotulos_v2.jpg", dpi=80); plt.show()

## 6. Treinamento v2
Mesma arquitetura da v1 (**YOLO26n**), para isolar o efeito dos dados. Se sobrar tempo, mude `TREINAR_S = True` para treinar também o **YOLO26s** (maior e mais preciso, porém mais lento no celular).

In [ ]:
from ultralytics import YOLO
import pandas as pd

TREINAR_S = False
HIP = dict(data="/content/dados/alagamento/data.yaml", epochs=70, imgsz=640, batch=16, patience=15,
           seed=42, deterministic=True, cos_lr=True, close_mosaic=10,
           hsv_h=0.015, hsv_s=0.6, hsv_v=0.5, fliplr=0.5, flipud=0.0, scale=0.5, mosaic=1.0,
           project=RUNS, exist_ok=True, plots=True)

ARQS = ["yolo26n.pt"] + (["yolo26s.pt"] if TREINAR_S else [])
for arq in ARQS:
    nome = arq.replace(".pt", "") + "_v2"
    ultimo = f"{RUNS}/{nome}/weights/last.pt"
    if os.path.exists(ultimo) and not os.path.exists(f"{RUNS}/{nome}/weights/best.pt.done"):
        print("Retomando", nome); YOLO(ultimo).train(resume=True)          # Colab caiu no meio: continua de onde parou
    else:
        YOLO(arq).train(name=nome, **HIP)
    open(f"{RUNS}/{nome}/weights/best.pt.done", "w").write("ok")

## 7. Escolha pela VALIDAÇÃO: v1 × v2
O teste só é usado depois, uma vez.

In [ ]:
candidatos = {"v1 (YOLO26n)": "modelos/v1_best.pt"}
for arq in ARQS:
    nome = arq.replace(".pt", "") + "_v2"
    candidatos[f"v2 ({arq[:-3]})"] = f"{RUNS}/{nome}/weights/best.pt"
val = {}
for rotulo, caminho in candidatos.items():
    m = YOLO(caminho).val(data=HIP["data"], split="val", imgsz=640, verbose=False)
    nomes = list(m.names.values())
    ap = {nomes[int(c)]: round(float(m.box.ap50[i]), 4) for i, c in enumerate(m.box.ap_class_index)}
    val[rotulo] = {"mAP50": round(float(m.box.map50), 4), "mAP50_95": round(float(m.box.map), 4),
                   "AP50_agua": ap.get("area_alagada"), "AP50_veiculo": ap.get("veiculo_ilhado"),
                   "AP50_pessoa": ap.get("pessoa_na_agua"), "ms": round(float(m.speed["inference"]), 2)}
tab_val = pd.DataFrame(val).T; display(tab_val)
melhor = max(val, key=lambda k: val[k]["mAP50_95"])
print("Melhor na validação:", melhor)

In [ ]:
os.makedirs("docs/resultados_v2", exist_ok=True)
if not melhor.startswith("v1"):
    shutil.copy2(candidatos[melhor], "modelos/best.pt")
    nome_run = Path(candidatos[melhor]).parent.parent
    for f in ["results.png", "labels.jpg", "val_batch0_pred.jpg"]:
        if (nome_run / f).exists(): shutil.copy2(nome_run / f, f"docs/resultados_v2/treino_{f}")
json.dump({"validacao": val, "escolhido": melhor, "hiperparametros": {k: v for k, v in HIP.items() if k not in ("project", "exist_ok", "plots")}},
          open("docs/resultados_v2/comparativo_v1_v2.json", "w"), indent=2, ensure_ascii=False)
print("modelos/best.pt agora é:", melhor)

## 8. Teste final: v1 e v2 no MESMO teste novo

In [ ]:
!python src/avaliar.py --modelo modelos/v1_best.pt --dados /content/dados/alagamento/data.yaml --saida docs/resultados_v2/v1_no_teste_v2
!python src/avaliar.py --modelo modelos/best.pt --dados /content/dados/alagamento/data.yaml --saida docs/resultados_v2

In [ ]:
def resumo(pasta):
    m = json.load(open(f"{pasta}/metricas.json")); r = json.load(open(f"{pasta}/nivel_risco.json"))
    pc = m["por_classe"]
    return {"mAP50": m["mAP50"], "mAP50_95": m["mAP50_95"],
            "AP50 água": pc.get("area_alagada", {}).get("mAP50"), "AP50 veículo": pc.get("veiculo_ilhado", {}).get("mAP50"),
            "AP50 pessoa": pc.get("pessoa_na_agua", {}).get("mAP50"),
            "recall alerta": r["alerta_recall"], "precisão alerta": r["alerta_precisao"], "acerto nível": r["acuracia_nivel"]}
final = pd.DataFrame({"v1": resumo("docs/resultados_v2/v1_no_teste_v2"), "escolhido (" + melhor + ")": resumo("docs/resultados_v2")}).T
final.to_csv("docs/resultados_v2/comparativo_teste.csv"); display(final)
from IPython.display import Image, display as show
show(Image("docs/resultados_v2/confusion_matrix_normalized.png", width=600))

## 9. Exemplos de resultado (teste novo)

In [ ]:
!rm -rf /content/pred_teste && python src/detectar.py --modelo modelos/best.pt --fonte /content/dados/alagamento/test/images --salvar /content/pred_teste > /dev/null
import glob
random.seed(7)
fotos = random.sample(sorted(glob.glob("/content/pred_teste/*")), 12)
fig, ax = plt.subplots(3, 4, figsize=(20, 13))
for a, f in zip(ax.flat, fotos):
    a.imshow(cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB)); a.axis("off")
plt.tight_layout(); plt.savefig("docs/img/exemplos_teste_v2.jpg", dpi=80); plt.show()

## 10. Exportar para o MVP e baixar

In [ ]:
!python src/exportar.py --modelo modelos/best.pt --imgsz 640 --metricas docs/resultados_v2/metricas.json
!rm -f /content/resultados_visao_v2.zip && zip -q -r /content/resultados_visao_v2.zip modelos/best.pt modelos/v1_best.pt docs web/model -x "docs/resultados_v2/val_teste/*" "docs/resultados_v2/v1_no_teste_v2/val_teste/*"
from google.colab import files
files.download("/content/resultados_visao_v2.zip")